# Bag-of-Words SL

Sample script to exemplify **instantiating** and **supervised-learning** on 
the bag-of-words dataset. 

In [ ]:
from pathlib import Path

import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.sft import BagOfWordsSFTConfig

repo_root = get_repo_base()
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
device

## Configure

In [ ]:
config = BagOfWordsSFTConfig.get_canonical(
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-sl-example",
    snr=0.01,
    num_words=15,
    num_samples=50_000,
    aux_words_ratio=0.5,
    prompt_length=128,
    filter_samples_above_n_tokens=384,
    word_decay_power=1.0,
    batch_size=64,
    eval_batch_size_multiple=4,
    lr_per_token=1.25e-7,
    backbone_lr_divisor=5.0,
    pad_to_multiple=8,
    train_epochs=12,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset Rsq target: {config.data.rsq:.4f}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

In [ ]:
state.run_training()

sft epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/97 [00:00<?, ?it/s]

epoch  0  train_corr_target=0.0008  train_corr_ground_truth=-0.0023  pred_norm=2.4118  val_corr_target=0.0013  val_corr_ground_truth=0.0147


sft epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

## Results

In [ ]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

In [ ]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()